In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-mpf qiskit-addon-utils scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Formule multi-prodotto per ridurre l'errore di Trotter

import TutorialFeedback from '@site/src/components/TutorialFeedback';






*Utilizzo stimato della QPU: Quattro minuti su un processore Heron r2 (NOTA: Questa è solo una stima. Il tempo di esecuzione potrebbe variare.)*

## Contesto
Questo tutorial dimostra come utilizzare una Formula Multi-Prodotto (MPF) per ottenere un errore di Trotter inferiore sull'osservabile rispetto a quello causato dal circuito di Trotter più profondo che effettivamente eseguiremo.
Le MPF riducono l'errore di Trotter della dinamica hamiltoniana attraverso una combinazione pesata di diverse esecuzioni di circuiti. Considerate il compito di trovare i valori di aspettazione di osservabili per lo stato quantistico
$\rho(t)=e^{-i H t} \rho(0) e^{i H t}$ con l'hamiltoniana $H$. Si possono utilizzare le Formule di Prodotto (PF) per approssimare l'evoluzione temporale $e^{-i H t}$ procedendo come segue:

- Scrivere l'hamiltoniana $H$ come $H=\sum_{a=1}^d F_a,$ dove $F_a$ sono operatori hermitiani tali che ciascun unitario corrispondente possa essere implementato in modo efficiente su un dispositivo quantistico.
- Approssimare i termini $F_a$ che non commutano tra loro.

Quindi, la PF del primo ordine (formula di Lie-Trotter) è:

$$S_1(t):=\prod_{a=1}^d e^{-i F_a t},$$

che ha un termine di errore quadratico $S_1(t)=e^{-i H t}+\mathcal{O}\left(t^{2}\right)$. Si possono anche utilizzare PF di ordine superiore (formule di Lie-Trotter-Suzuki), che convergono più rapidamente e sono definite ricorsivamente come:

$$S_2(t):=\prod_{a=1}^d e^{-i F_a t/2}\prod_{a=1}^d e^{-i F_a t/2}$$

$$S_{2 \chi}(t):= S_{2 \chi -2}(s_{\chi}t)^2 S_{2 \chi -2}((1-4s_{\chi})t)S_{2 \chi -2}(s_{\chi}t)^2,$$

dove $\chi$ è l'ordine della PF simmetrica e $s_p = \left( 4 - 4^{1/(2p-1)} \right)^{-1}$. Per evoluzioni temporali lunghe, si può dividere l'intervallo di tempo $t$ in $k$ intervalli, chiamati passi di Trotter, di durata $t/k$ e approssimare l'evoluzione temporale in ciascun intervallo con una formula di prodotto di ordine $\chi$ $S_{\chi}$. Quindi, la PF di ordine $\chi$ per l'operatore di evoluzione temporale su $k$ passi di Trotter è:

$$ S_{\chi}^{k}(t) = \left[ S_{\chi} \left( \frac{t}{k} \right)\right]^k = e^{-i H t}+O\left(t \left( \frac{t}{k} \right)^{\chi} \right)$$

dove il termine di errore diminuisce con il numero di passi di Trotter $k$ e l'ordine $\chi$ della PF.

Dato un intero $k \geq 1$ e una formula di prodotto $S_{\chi}(t)$, lo stato approssimato evoluto nel tempo $\rho_k(t)$ può essere ottenuto da $\rho_0$ applicando $k$ iterazioni della formula di prodotto $S_{\chi}\left(\frac{t}{k}\right)$.

$$
\rho_k(t)=S_{\chi}\left(\frac{t}{k}\right)^k \rho_0 S_{\chi}\left(\frac{t}{k}\right)^{-k}
$$

$\rho_k(t)$ è un'approssimazione di $\rho(t)$ con l'errore di approssimazione di Trotter ||$\rho_k(t)-\rho(t) ||$. Se consideriamo una combinazione lineare di approssimazioni di Trotter di $\rho(t)$:

$$
\mu(t) = \sum_{j}^{l} x_j \rho^{k_j}_{j}\left(\frac{t}{k_j}\right) + \text{un certo errore di Trotter residuo} \, ,
$$

dove $x_j$ sono i nostri coefficienti di ponderazione, $\rho^{k_j}_j$ è la matrice densità corrispondente allo stato puro ottenuto evolvendo lo stato iniziale con la formula di prodotto, $S^{k_j}_{\chi}$, che coinvolge $k_j$ passi di Trotter, e $j \in {1, ..., l}$ indicizza il numero di PF che compongono la MPF. Tutti i termini in $\mu(t)$ utilizzano la stessa formula di prodotto $S_{\chi}(t)$ come base.
L'obiettivo è migliorare ||$\rho_k(t)-\rho(t) \|$ trovando $\mu(t)$ con $\|\mu(t)-\rho(t)\|$ ancora più basso.

* $\mu(t)$ non deve necessariamente essere uno stato fisico poiché $x_i$ non deve essere positivo. L'obiettivo qui è minimizzare l'errore nel valore di aspettazione degli osservabili e non trovare una sostituzione fisica per $\rho(t)$.
* $k_j$ determina sia la profondità del circuito che il livello di approssimazione di Trotter. Valori più piccoli di $k_j$ portano a circuiti più brevi, che incorrono in minori errori di circuito ma saranno un'approssimazione meno accurata dello stato desiderato.

La chiave qui è che l'errore di Trotter residuo dato da $\mu(t)$ è più piccolo dell'errore di Trotter che si otterrebbe semplicemente utilizzando il valore più grande di $k_j$.

Potete vedere l'utilità di questo da due prospettive:

1. Per un budget fisso di passi di Trotter che potete eseguire, potete ottenere risultati con un errore di Trotter che è complessivamente più piccolo.
2. Dato un numero target di passi di Trotter che è troppo grande da eseguire, potete utilizzare la MPF per trovare una collezione di circuiti a profondità inferiore da eseguire che risulta in un errore di Trotter simile.
## Requisiti
Prima di iniziare questo tutorial, assicuratevi di avere installato quanto segue:

* Qiskit SDK v1.0 o successivo, con supporto per la [visualizzazione](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.22 o successivo (`pip install qiskit-ibm-runtime`)
* MPF Qiskit addons (`pip install qiskit_addon_mpf`)
* Qiskit addons utils (`pip install qiskit_addon_utils`)
* Libreria Quimb (`pip install quimb`)
* Libreria Qiskit Quimb (`pip install qiskit-quimb`)
* Numpy v0.21 per la compatibilità tra i pacchetti (`pip install numpy==0.21`)
## Parte I. Esempio su piccola scala
### Esplorare la stabilità della MPF
Non c'è una restrizione ovvia sulla scelta del numero di passi di Trotter $k_j$ che compongono lo stato MPF $\mu(t)$. Tuttavia, questi devono essere scelti con attenzione per evitare instabilità nei valori di aspettazione risultanti calcolati da $\mu(t)$. Una buona regola generale è impostare il passo di Trotter più piccolo $k_{\text{min}}$ in modo che $t/k_{\text{min}} \lt 1$. Se volete saperne di più su questo e su come scegliere gli altri valori $k_j$, fate riferimento alla guida [Come scegliere i passi di Trotter per una MPF](https://qiskit.github.io/qiskit-addon-mpf/how_tos/choose_trotter_steps.html).

Nell'esempio seguente esploriamo la stabilità della soluzione MPF calcolando il valore di aspettazione della magnetizzazione per un intervallo di tempi utilizzando diversi stati evoluti nel tempo. Nello specifico, confrontiamo i valori di aspettazione calcolati da ciascuna delle evoluzioni temporali approssimate implementate con i corrispondenti passi di Trotter e i vari modelli MPF (coefficienti statici e dinamici) con i valori esatti dell'osservabile evoluto nel tempo. Innanzitutto, definiamo i parametri per le formule di Trotter e i tempi di evoluzione

In [1]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from functools import partial
from copy import deepcopy

from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.synthesis import SuzukiTrotter
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import XXPlusYYGate
from qiskit.transpiler.passes.optimization.collect_and_collapse import (
    CollectAndCollapse,
    collect_using_filter_function,
    collapse_to_operation,
)

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

from qiskit_addon_utils.problem_generators import (
    generate_xyz_hamiltonian,
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth
from qiskit_addon_mpf.static import setup_static_lse
from qiskit_addon_mpf.dynamic import setup_dynamic_lse
from qiskit_addon_mpf.costs import (
    setup_exact_problem,
    setup_sum_of_squares_problem,
    setup_frobenius_problem,
)
from qiskit_addon_mpf.backends.tenpy_layers import (
    LayerModel,
    LayerwiseEvolver,
)
from qiskit_addon_mpf.backends.tenpy_tebd import MPOState, MPS_neel_state

from scipy.linalg import expm

# Suppress TeNPy's `unit_cell_width` future-API warning. The default
# (`unit_cell_width=len(sites)`) is correct for Chain lattices, which is what
# `CouplingMap.from_line(...)` produces here, so the warning is informational.
warnings.filterwarnings(
    "ignore",
    message=r".*unit_cell_width.*",
    category=UserWarning,
)


# --- Helper: collect XX + YY rotations into a single gate ---
def filter_function(node):
    return node.op.name in {"rxx", "ryy"}


collect_function = partial(
    collect_using_filter_function,
    filter_function=filter_function,
    split_blocks=True,
    min_block_size=1,
)


def collapse_to_xx_plus_yy(block):
    param = 0.0
    for node in block.data:
        param += node.operation.params[0]
    return XXPlusYYGate(param)


collapse_function = partial(
    collapse_to_operation,
    collapse_function=collapse_to_xx_plus_yy,
)

pm = PassManager()
pm.append(CollectAndCollapse(collect_function, collapse_function))

Per questo esempio utilizzeremo lo stato di Neel come stato iniziale $\vert \text{Neel} \rangle = \vert 0101...01 \rangle$ e il modello di Heisenberg su una linea di 10 siti per l'hamiltoniana che governa l'evoluzione temporale

$$
\hat{\mathcal{H}}_{Heis} = J \sum_{i=1}^{L-1} \left(X_i X_{(i+1)}+Y_i Y_{(i+1)}+ Z_i Z_{(i+1)} \right) \, ,
$$

dove $J$ è l'intensità dell'accoppiamento per i bordi tra i vicini più prossimi.

In [2]:
L = 10

# Generate coupling map and Hamiltonian
coupling_map = CouplingMap.from_line(L, bidirectional=False)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(1.0, 1.0, 1.0),
    ext_magnetic_field=(0.0, 0.0, 0.0),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


In [3]:
# Observable: ZZ on the middle pair of qubits
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)
print(observable)

SparsePauliOp(['IIIIZZIIII'],
              coeffs=[1.+0.j])


In [4]:
# MPF parameters
mpf_trotter_steps = [1, 2, 4]
order = 2
symmetric = False

trotter_times = np.arange(0.5, 1.55, 0.1)
exact_evolution_times = np.arange(trotter_times[0], 1.55, 0.05)

#### Build Trotter circuits

We create the circuits implementing the approximate Trotter time-evolutions for each time point and each Trotter step count. The `CollectAndCollapse` pass defined in the Setup section collects XX and YY rotations into single XX+YY gates, to prepare for more efficient tensor-network simulation later.

In [5]:
# Initial Neel state preparation
initial_state_circ = QuantumCircuit(L)
initial_state_circ.x([i for i in range(L) if i % 2 != 0])


all_circs = []
for total_time in trotter_times:
    mpf_trotter_circs = [
        generate_time_evolution_circuit(
            hamiltonian,
            time=total_time,
            synthesis=SuzukiTrotter(reps=num_steps, order=order),
        )
        for num_steps in mpf_trotter_steps
    ]

    mpf_trotter_circs = pm.run(
        mpf_trotter_circs
    )  # Collect XX and YY into XX + YY

    mpf_circuits = [
        initial_state_circ.compose(circuit) for circuit in mpf_trotter_circs
    ]
    all_circs.append(mpf_circuits)

In [6]:
mpf_circuits[-1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/c7ee61e7-0.avif" alt="Output of the previous code cell" />

Quindi creiamo i circuiti che implementano le evoluzioni temporali approssimate di Trotter.

In [7]:
aer_sim = AerSimulator()
pm_sim = generate_preset_pass_manager(backend=aer_sim, optimization_level=3)

isa_circs_all_times = [
    pm_sim.run([deepcopy(c) for c in mpf_circuits])
    for mpf_circuits in all_circs
]

### Step 3: Execute using Qiskit primitives

For the small-scale example we run the ISA-lowered Trotter circuits through the `EstimatorV2` primitive backed by Aer. Doing so gives us a *noiseless* reference value for each $(k_j, t)$ pair &mdash; these are the $\langle A \rangle_{k_j}(t)$ values that the MPF will combine in Step 4. We sweep over evolution times so that we can later plot the full time-series curve of each individual product formula and of the MPF.

In [8]:
estimator = Estimator(mode=aer_sim)

mpf_expvals_all_times, mpf_stds_all_times = [], []
for isa_circuits in isa_circs_all_times:
    result = estimator.run(
        [(circuit, observable) for circuit in isa_circuits], precision=0.005
    ).result()
    mpf_expvals_all_times.append([res.data.evs for res in result])
    mpf_stds_all_times.append([res.data.stds for res in result])

![Output of the previous code cell](../docs/images/tutorials/multi-product-formula/extracted-outputs/92dc20a7-0.avif)

Successivamente, calcoliamo i valori di aspettazione evoluti nel tempo dai circuiti di Trotter.

In [9]:
exact_expvals = []
for t in exact_evolution_times:
    exp_H = expm(-1j * t * hamiltonian.to_matrix())
    initial_state = Statevector(initial_state_circ).data
    time_evolved_state = exp_H @ initial_state

    exact_obs = (
        time_evolved_state.conj()
        @ observable.to_matrix()
        @ time_evolved_state
    ).real
    exact_expvals.append(exact_obs)

Calcoliamo anche i valori di aspettazione esatti per il confronto.

In [10]:
lse = setup_static_lse(mpf_trotter_steps, order=order, symmetric=symmetric)

#### Coefficienti MPF statici
Le MPF statiche sono quelle in cui i valori $x_j$ non dipendono dal tempo di evoluzione, $t$. Consideriamo la PF di ordine $\chi = 1$ con $k_j$ passi di Trotter, questa può essere scritta come:

$$ S_1^{k_j}\left( \frac{t}{k_j} \right)=e^{-i H t}+ \sum_{n=1}^{\infty} A_n \frac{t^{n+1}}{k_j^n}  $$

dove $A_n$ sono matrici che dipendono dai commutatori dei termini $F_a$ nella decomposizione dell'hamiltoniana. È importante notare che $A_n$ stessi sono indipendenti dal tempo e dal numero di passi di Trotter $k_j$. Pertanto, è possibile cancellare i termini di errore di ordine inferiore che contribuiscono a $\mu(t)$ con una scelta attenta dei pesi $x_j$ della combinazione lineare. Per cancellare l'errore di Trotter per i primi $l-1$ termini (questi daranno i contributi maggiori poiché corrispondono al numero inferiore di passi di Trotter) nell'espressione per $\mu(t)$, i coefficienti $x_j$ devono soddisfare le seguenti equazioni:

$$ \sum_{j=1}^l x_j = 1 $$
$$ \sum_{j=1}^{l-1} \frac{x_j}{k_j^{n}} = 0 $$

con $n=0, ... l-2$. La prima equazione garantisce che non ci sia bias nello stato costruito $\mu(t)$, mentre la seconda equazione assicura la cancellazione degli errori di Trotter. Per PF di ordine superiore, la seconda equazione diventa $ \sum_{j=1}^{l-1} \frac{x_j}{k_j^{\eta}} = 0 $ dove $\eta = \chi + 2n$ per PF simmetriche e $\eta = \chi + n$ altrimenti, con $n=0, ..., l-2$. L'errore risultante (Rif. [\[1\]](#references),[\[2\]](#references)) è quindi

$$ \epsilon = \mathcal{O} \left( \frac{t^{l+1}}{k_1^l} \right).$$

Determinare i coefficienti MPF statici per un dato insieme di valori $k_j$ equivale a risolvere il sistema lineare di equazioni definito dalle due equazioni precedenti per le variabili $x_j$: $Ax=b$. Dove $x$ sono i nostri coefficienti di interesse, $A$ è una matrice che dipende da $k_j$ e dal tipo di PF che utilizziamo ($S$), e $b$ è un vettore di vincoli. Specificamente:

$$A_{0,j} = 1 $$
$$A_{i>0,j} = k_{j}^{-(\chi + s(i-1))}$$
$$b_0 = 1$$
$$b_{i>0} = 0 $$

dove $\chi$ è l'``order``, $s$ è $2$ se ``symmetric`` è ``True`` e $1$ altrimenti, $k_{j}$ sono i ``trotter_steps``, e $x$ sono le variabili da risolvere. Gli indici $i$ e $j$ partono da $0$. Possiamo anche visualizzare questo in forma matriciale:

$$
A =
\begin{bmatrix}
A_{0,0} & A_{0,1} & A_{0,2} & ... \\
A_{1,0} & A_{1,1} & A_{1,2} & ...  \\
A_{2,0} & A_{2,1} & A_{2,2} & ...  \\
... & ... & ... & ...
\end{bmatrix} =
\begin{bmatrix}
1 & 1 & 1 & ... \\
k_{0}^{-(\chi + s(1-1))} & k_{1}^{-(\chi + s(1-1))} & k_{2}^{-(\chi + s(1-1))} & ... \\
k_{0}^{-(\chi + s(2-1))} & k_{1}^{-(\chi + s(2-1))} & k_{2}^{-(\chi + s(2-1))} & ... \\
... & ... & ... & ...
\end{bmatrix}
$$

e

$$
b =
\begin{bmatrix}
b_{0} \\
b_{1} \\
b_{2}  \\
...
\end{bmatrix} =
\begin{bmatrix}
1 \\
0 \\
0  \\
...
\end{bmatrix}
$$

Per maggiori dettagli, fate riferimento alla documentazione del Sistema Lineare di Equazioni ([LSE](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.LSE.html)).

Possiamo trovare una soluzione per $x$ analiticamente come $x = A^{-1}b$; vedete ad esempio i Rif. [\[1\]](#references) o [\[2\]](#references).
Tuttavia, questa soluzione esatta può essere "mal condizionata", risultando in norme L1 molto grandi dei nostri coefficienti, $x$, che possono portare a cattive prestazioni della MPF.
Invece, si può anche ottenere una soluzione approssimata che minimizza la norma L1 di $x$ al fine di tentare di ottimizzare il comportamento della MPF.
##### Impostare l'LSE
Ora che abbiamo scelto i nostri valori $k_j$, dobbiamo prima costruire l'LSE, $Ax=b$ come spiegato sopra.
La matrice $A$ dipende non solo da $k_j$ ma anche dalla nostra scelta di PF, in particolare dal suo _ordine_.
Inoltre, potreste tenere conto se la PF è simmetrica o no (vedete [\[1\]](#references)) impostando `symmetric=True/False`.
Tuttavia, questo non è richiesto, come mostrato dal Rif. [\[2\]](#references).

In [11]:
lse.A

array([[1.      , 1.      , 1.      ],
       [1.      , 0.25    , 0.0625  ],
       [1.      , 0.125   , 0.015625]])

In [12]:
lse.b

array([1., 0., 0.])

With the LSE in hand, we solve for the static coefficients $x_j$ via `lse.solve()` (this is the direct $x = A^{-1}b$ solution).

In [13]:
mpf_coeffs = lse.solve()
print(
    f"The static coefficients associated with the ansatze are: {mpf_coeffs}"
)

The static coefficients associated with the ansatze are: [ 0.04761905 -0.57142857  1.52380952]


Mentre il vettore di vincoli $b$ ha i seguenti elementi:
$$ b_{0} = 1 $$
$$ b_1 = b_2 = 0 $$

Quindi,

$$
b =
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

E similmente in `lse`:

In [14]:
model_exact, coeffs_exact = setup_exact_problem(lse)
model_exact.solve()
print(coeffs_exact.value)

[ 0.04761905 -0.57142857  1.52380952]


In [15]:
print(
    "L1 norm of the exact coefficients:",
    np.linalg.norm(coeffs_exact.value, ord=1),
)

L1 norm of the exact coefficients: 2.1428571428556378


L'oggetto `lse` ha metodi per trovare i coefficienti statici $x_j$ che soddisfano il sistema di equazioni.

In [16]:
model_approx, coeffs_approx = setup_sum_of_squares_problem(
    lse, max_l1_norm=1.5
)
model_approx.solve()
print(coeffs_approx.value)
print(
    "L1 norm of the approximate coefficients:",
    np.linalg.norm(coeffs_approx.value, ord=1),
)

[-1.10294118e-03 -2.48897059e-01  1.25000000e+00]
L1 norm of the approximate coefficients: 1.5


#### Dynamic MPF coefficients

The static MPF cancels Trotter-error terms in a Hamiltonian- and state-agnostic way, so it does not necessarily produce the smallest possible approximation error for a given Hamiltonian and initial state. The dynamic MPF (Refs. [\[2\]](#references), [\[3\]](#references)) instead finds time-dependent coefficients $x_i(t)$ that minimize the Frobenius-norm distance $\|\rho(t) - \mu^D(t)\|_F^2$ at each time $t$. As shown in the Background, this requires the overlap matrix $M_{ij}(t)$ between Trotter-evolved states and the overlap $L_i(t)$ with the exact state &mdash; both of which we estimate using tensor-network (TeNPy) backends in `qiskit_addon_mpf`.

To set up the dynamic LSE we need three ingredients:

1. An **approximate evolver factory** that the addon will run for each $k_j$ to produce $\rho_{k_j}(t)$ as an MPS/MPO. We build it from the layered structure of the order-$2$ Trotter circuit (one layer per `slice_by_depth`), wrapped as a `LayerwiseEvolver` with TeNPy truncation parameters.
2. An **exact evolver factory** that produces a high-accuracy reference $\rho(t)$. We use a small-time-step fourth-order Suzuki-Trotter circuit (`dt=0.1`, `order=4`) as a proxy for exact evolution.
3. An **identity factory** and an **initial-state MPS** that seed the TeNPy simulation.

The cell below constructs the approximate evolver factory.

In [17]:
# Create approximate time-evolution circuits
single_2nd_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=order)
)
single_2nd_order_circ = pm.run(single_2nd_order_circ)  # collect XX and YY

# Find layers in the circuit
layers = slice_by_depth(single_2nd_order_circ, max_slice_depth=1)

# Create tensor network models
models = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz") for layer in layers
]

# Create the time-evolution object
approx_factory = partial(
    LayerwiseEvolver,
    layers=models,
    options={
        "preserve_norm": False,
        "trunc_params": {
            "chi_max": 64,
            "svd_min": 1e-8,
            "trunc_cut": None,
        },
        "max_delta_t": 2,
    },
)

##### Ottimizzare per $x$ usando un modello esatto
Alternativamente al calcolo di $x=A^{-1}b$, potete anche utilizzare [setup_exact_model](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.setup_exact_model.html) per costruire un'istanza di [cvxpy.Problem](https://www.cvxpy.org/api_reference/cvxpy.problems.html#cvxpy.Problem) che utilizza l'LSE come vincoli e la cui soluzione ottimale produrrà $x$.

Nella prossima sezione, sarà chiaro perché esiste questa interfaccia.

In [18]:
single_4th_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=4)
)
single_4th_order_circ = pm.run(single_4th_order_circ)
exact_model_layers = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz")
    for layer in slice_by_depth(single_4th_order_circ, max_slice_depth=1)
]

exact_factory = partial(
    LayerwiseEvolver,
    layers=exact_model_layers,
    dt=0.1,
    options={
        "preserve_norm": False,
        "trunc_params": {
            "chi_max": 64,
            "svd_min": 1e-8,
            "trunc_cut": None,
        },
        "max_delta_t": 2,
    },
)

Finally, we define an `identity_factory` that yields the initial MPO state and prepare the Néel initial state as an MPS that matches the lattice used by the layered Trotter model.

In [19]:
def identity_factory():
    return MPOState.initialize_from_lattice(models[0].lat, conserve=True)


mps_initial_state = MPS_neel_state(models[0].lat)

Come indicatore se una MPF costruita con questi coefficienti produrrà buoni risultati, possiamo utilizzare la norma L1 (vedete anche Rif. [\[1\]](#references)).

In [20]:
mpf_dynamic_coeffs_list = []
for t in trotter_times:
    print(f"Computing dynamic coefficients for time={t}")
    lse = setup_dynamic_lse(
        mpf_trotter_steps,
        t,
        identity_factory,
        exact_factory,
        approx_factory,
        mps_initial_state,
    )
    problem, coeffs = setup_frobenius_problem(lse)
    try:
        problem.solve()
        mpf_dynamic_coeffs_list.append(coeffs.value)
    except Exception as error:
        mpf_dynamic_coeffs_list.append(np.zeros(len(mpf_trotter_steps)))
        print(error, "Calculation Failed for time", t)
    print("")

Computing dynamic coefficients for time=0.5

Computing dynamic coefficients for time=0.6

Computing dynamic coefficients for time=0.7

Computing dynamic coefficients for time=0.7999999999999999

Computing dynamic coefficients for time=0.8999999999999999

Computing dynamic coefficients for time=0.9999999999999999

Computing dynamic coefficients for time=1.0999999999999999

Computing dynamic coefficients for time=1.1999999999999997

Computing dynamic coefficients for time=1.2999999999999998

Computing dynamic coefficients for time=1.4

Computing dynamic coefficients for time=1.4999999999999998



#### Combine Trotter expectation values with the MPF coefficients

Now we evaluate $\langle A \rangle_{\text{MPF}}(t) = \sum_j x_j \, \langle A \rangle_{k_j}(t)$ for each set of coefficients (static-exact, static-approximate, and dynamic), propagate the per-circuit standard errors, and plot the resulting time series against the exact-diagonalization curve.

In [21]:
sym = {1: "^", 2: "s", 4: "p"}
# Get expectation values at all times for each Trotter step
for k, step in enumerate(mpf_trotter_steps):
    trotter_curve, trotter_curve_error = [], []
    for trotter_expvals, trotter_stds in zip(
        mpf_expvals_all_times, mpf_stds_all_times
    ):
        trotter_curve.append(trotter_expvals[k])
        trotter_curve_error.append(trotter_stds[k])

    plt.errorbar(
        trotter_times,
        trotter_curve,
        yerr=trotter_curve_error,
        alpha=0.5,
        markersize=4,
        marker=sym[step],
        color="grey",
        label=f"{mpf_trotter_steps[k]} Trotter steps",
    )

# Get expectation values at all times for the static MPF with exact coeffs
exact_mpf_curve, exact_mpf_curve_error = [], []
for trotter_expvals, trotter_stds in zip(
    mpf_expvals_all_times, mpf_stds_all_times
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(coeffs_exact.value, trotter_stds)
            ]
        )
    )
    exact_mpf_curve_error.append(mpf_std)
    exact_mpf_curve.append(trotter_expvals @ coeffs_exact.value)

plt.errorbar(
    trotter_times,
    exact_mpf_curve,
    yerr=exact_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Static MPF - Exact",
    color="purple",
)


# Get expectation values at all times for the static MPF with approximate coeffs
approx_mpf_curve, approx_mpf_curve_error = [], []
for trotter_expvals, trotter_stds in zip(
    mpf_expvals_all_times, mpf_stds_all_times
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(coeffs_approx.value, trotter_stds)
            ]
        )
    )
    approx_mpf_curve_error.append(mpf_std)
    approx_mpf_curve.append(trotter_expvals @ coeffs_approx.value)

plt.errorbar(
    trotter_times,
    approx_mpf_curve,
    yerr=approx_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Static MPF - Approx",
    color="orange",
)


# Get expectation values at all times for the dynamic MPF
dynamic_mpf_curve, dynamic_mpf_curve_error = [], []
for trotter_expvals, trotter_stds, dynamic_coeffs in zip(
    mpf_expvals_all_times, mpf_stds_all_times, mpf_dynamic_coeffs_list
):
    mpf_std = np.sqrt(
        sum(
            [
                (coeff**2) * (std**2)
                for coeff, std in zip(dynamic_coeffs, trotter_stds)
            ]
        )
    )
    dynamic_mpf_curve_error.append(mpf_std)
    dynamic_mpf_curve.append(trotter_expvals @ dynamic_coeffs)

plt.errorbar(
    trotter_times,
    dynamic_mpf_curve,
    yerr=dynamic_mpf_curve_error,
    markersize=4,
    marker="o",
    label="Dynamic MPF",
    color="pink",
)


# Exact expectation values
plt.plot(
    exact_evolution_times,
    exact_expvals,
    color="red",
    linestyle="--",
    label="Exact time-evolution",
)

plt.title(f"$\\langle Z_{{{L//2-1}}} Z_{{{L//2}}} \\rangle$ vs time")
plt.xlabel("Time")
plt.ylabel("Expectation Value")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/35042576-0.avif" alt="Output of the previous code cell" />

##### Ottimizzare per $x$ usando un modello approssimato
Potrebbe accadere che la norma L1 per l'insieme scelto di valori $k_j$ sia ritenuta troppo alta.
Se questo è il caso e non potete scegliere un diverso insieme di valori $k_j$, potete utilizzare una soluzione approssimata all'LSE invece di una esatta.

Per farlo, utilizzate semplicemente [setup_approximate_model](https://qiskit.github.io/qiskit-addon-mpf/stubs/qiskit_addon_mpf.static.setup_approximate_model.html) per costruire una diversa istanza di [cvxpy.Problem](https://www.cvxpy.org/api_reference/cvxpy.problems.html#cvxpy.Problem), che vincola la norma L1 a una soglia scelta minimizzando al contempo la differenza tra $Ax$ e $b$.

In [22]:
# -------------------------Step 1-------------------------
L = 50
coupling_map = CouplingMap.from_line(L, bidirectional=False)

# XXZ Hamiltonian with random couplings (Ref. [3])
np.random.seed(0)
even_edges = list(coupling_map.get_edges())[::2]
odd_edges = list(coupling_map.get_edges())[1::2]

Js = np.random.uniform(0.5, 1.5, size=L)
hamiltonian = SparsePauliOp(Pauli("I" * L))
for i, edge in enumerate(even_edges + odd_edges):
    hamiltonian += SparsePauliOp.from_sparse_list(
        [
            ("XX", (edge), 2 * Js[i]),
            ("YY", (edge), 2 * Js[i]),
            ("ZZ", (edge), 4 * Js[i]),
        ],
        num_qubits=L,
    )

observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)

total_time = 3
mpf_trotter_steps = [3, 4, 6]
order = 2
symmetric = True

# Static coefficients
lse = setup_static_lse(mpf_trotter_steps, order=order, symmetric=symmetric)
mpf_coeffs = lse.solve()
print(f"Static coefficients: {mpf_coeffs}")
print(f"L1 norm: {np.linalg.norm(mpf_coeffs, ord=1)}")

model_approx, coeffs_approx = setup_sum_of_squares_problem(
    lse, max_l1_norm=2.0
)
model_approx.solve()
print(f"Approximate coefficients: {coeffs_approx.value}")
print(f"L1 norm (approx): {np.linalg.norm(coeffs_approx.value, ord=1)}")

# -------------------------Dynamic coefficients-------------------------
single_2nd_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=order)
)
single_2nd_order_circ = pm.run(single_2nd_order_circ)

layers = slice_by_depth(single_2nd_order_circ, max_slice_depth=1)
models = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz") for layer in layers
]

approx_factory = partial(
    LayerwiseEvolver,
    layers=models,
    options={
        "preserve_norm": False,
        "trunc_params": {"chi_max": 64, "svd_min": 1e-8, "trunc_cut": None},
        "max_delta_t": 4,
    },
)

single_4th_order_circ = generate_time_evolution_circuit(
    hamiltonian, time=1.0, synthesis=SuzukiTrotter(reps=1, order=4)
)
single_4th_order_circ = pm.run(single_4th_order_circ)
exact_model_layers = [
    LayerModel.from_quantum_circuit(layer, conserve="Sz")
    for layer in slice_by_depth(single_4th_order_circ, max_slice_depth=1)
]

exact_factory = partial(
    LayerwiseEvolver,
    layers=exact_model_layers,
    dt=0.1,
    options={
        "preserve_norm": False,
        "trunc_params": {"chi_max": 64, "svd_min": 1e-8, "trunc_cut": None},
        "max_delta_t": 3,
    },
)


def identity_factory():
    return MPOState.initialize_from_lattice(models[0].lat, conserve=True)


mps_initial_state = MPS_neel_state(models[0].lat)

print(f"Computing dynamic coefficients for time={total_time}")
lse_dyn = setup_dynamic_lse(
    mpf_trotter_steps,
    total_time,
    identity_factory,
    exact_factory,
    approx_factory,
    mps_initial_state,
)
problem, coeffs_dyn = setup_frobenius_problem(lse_dyn)
try:
    problem.solve()
    mpf_dynamic_coeffs = coeffs_dyn.value
except Exception as error:
    mpf_dynamic_coeffs = np.zeros(len(mpf_trotter_steps))
    print(error, "Calculation Failed")

# -------------------------Step 1 (cont): Build circuits-------------------------
mpf_circuits = []
for k in mpf_trotter_steps:
    circuit = QuantumCircuit(L)
    circuit.x([i for i in range(L) if i % 2])
    trotter_circ = generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=k, order=order),
        time=total_time,
    )
    circuit.compose(trotter_circ, qubits=range(L), inplace=True)
    mpf_circuits.append(circuit)

# Baseline "single deep circuit" comparison run with k=10 Trotter steps.
# Its two-qubit depth is deeper than the deepest MPF constituent (k_max=6) plus
# the overhead of running multiple circuits, pushing it into the noise-limited
# regime where MPF is expected to outperform. It does NOT target the MPF's effective
# Trotter error (which would require many more steps).
comp_circuit = QuantumCircuit(L)
comp_circuit.x([i for i in range(L) if i % 2])
trotter_circ = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=10, order=order),
    time=total_time,
)
comp_circuit.compose(trotter_circ, qubits=range(L), inplace=True)
mpf_circuits.append(comp_circuit)

Static coefficients: [ 0.42857143 -1.82857143  2.4       ]
L1 norm: 4.65714285714286
Approximate coefficients: [-0.4942491   0.40206845  1.09218065]
L1 norm (approx): 1.9884981979026675
Computing dynamic coefficients for time=3


We now optimize the circuits for the chosen backend. We use the Qiskit preset pass manager at `optimization_level=3`, which automatically selects a good set of physical qubits and routes each circuit onto the device topology.

In [23]:
# -------------------------Step 2-------------------------
service = QiskitRuntimeService()
# backend = service.least_busy(operational=True, simulator=False, min_num_qubits=L)
backend = service.backend("ibm_fez")
print(backend)

transpiler = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)
transpiled_circuits = [transpiler.run(circ) for circ in mpf_circuits]

isa_observables = [
    observable.apply_layout(circ.layout) for circ in transpiled_circuits
]

<IBMBackend('ibm_fez')>


Notate che avete completa libertà su come risolvere questo problema di ottimizzazione, il che significa che potete cambiare il risolutore di ottimizzazione, le sue soglie di convergenza e così via.
Consultate la guida relativa su [Come utilizzare il modello approssimato](https://qiskit.github.io/qiskit-addon-mpf/how_tos/using_approximate_model.html).
#### Coefficienti MPF dinamici
Nella sezione precedente, abbiamo introdotto una MPF statica che migliora l'approssimazione di Trotter standard. Tuttavia, questa versione statica non minimizza necessariamente l'errore di approssimazione. Concretamente, la MPF statica, indicata $\mu^S(t)$, non è la proiezione ottimale di $\rho(t)$ sul sottospazio spannato dagli stati della formula di prodotto ${\rho_{k_i}(t)}_{i=1}^r$.

Per affrontare questo, consideriamo una MPF dinamica (introdotta nel Rif. [\[2\]](#references) e dimostrata sperimentalmente nel Rif. [\[3\]](#references)) che minimizza l'errore di approssimazione nella norma di Frobenius. Formalmente, ci concentriamo sulla minimizzazione di

$$
\|\rho(t) - \mu^D(t)\|_F^2 \;=\; \mathrm{Tr}\bigl[ \left( \rho(t) - \mu^D(t)\right)^2 \bigr],
$$

rispetto ad alcuni coefficienti $x_i(t)$ ad ogni tempo $t$. Il proiettore *ottimale* nella norma di Frobenius è quindi $\mu^D(t) \;=\; \sum_{i=1}^r x_i(t)\,\rho_{k_i}(t)$, e chiamiamo $\mu^D(t)$ la MPF *dinamica*. Sostituendo le definizioni sopra:

$$
\|\rho(t) - \mu^D(t)\|_F^2
\;=\; \\
= \mathrm{Tr}\bigl[ \left( \rho(t) - \mu^D(t)\right)^2 \bigr]
\;=\; \\
= \mathrm{Tr}\bigl[ \left( \rho(t) - \sum_{i=1}^r x_i(t)\,\rho_{k_i}(t) \right) \left(  \rho(t) - \sum_{j=1}^r x_j(t)\,\rho_{k_j}(t) \right) \bigr]
\;=\; \\
= 1 \;+\; \sum_{i,j=1}^r M_{i,j}(t)\,x_i(t)\,x_j(t)
\;-\;
2 \sum_{i=1}^r L_i^{\mathrm{exact}}(t)\,x_i(t),
$$

dove $M_{i,j}(t)$ è la *matrice di Gram*, definita da

$$
M_{i,j}(t) \;=\; \mathrm{Tr}\bigl[\rho_{k_i}(t)\,\rho_{k_j}(t)\bigr]
\;=\;
\bigl|\langle \psi_{\mathrm{in}} \!\mid S\bigl(t/k_i\bigr)^{-k_i}\,S\bigl(t/k_j\bigr)^{k_j} \!\mid \psi_{\mathrm{in}} \rangle \bigr|^2.
$$

e

$$
L_i^{\mathrm{exact}}(t) = \mathrm{Tr}[\rho(t)\,\rho_{k_i}(t)]
$$

rappresenta la sovrapposizione tra lo stato esatto $\rho(t)$ e ciascuna approssimazione della formula di prodotto $\rho_{k_i}(t)$. In scenari pratici, queste sovrapposizioni possono essere misurate solo approssimativamente a causa del rumore o dell'accesso parziale a $\rho(t)$.

Qui, $\lvert\psi_{\mathrm{in}}\rangle$ è lo stato iniziale, e $S(\cdot)$ è l'operazione applicata nella formula di prodotto. Scegliendo i coefficienti $x_i(t)$ che minimizzano questa espressione (e gestendo i dati di sovrapposizione approssimati quando $\rho(t)$ non è completamente noto), otteniamo la "migliore" (in senso della norma di Frobenius) approssimazione dinamica di $\rho(t)$ all'interno del sottospazio MPF. Le quantità $L_i(t)$ e $M_{i,j}(t)$ possono essere calcolate in modo efficiente utilizzando metodi di reti tensoriali [\[3\]](#references). L'addon MPF di Qiskit fornisce diversi "backend" per eseguire il calcolo. L'esempio seguente mostra il modo più flessibile per farlo, e la documentazione del [backend basato su layer TeNPy](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.backends.tenpy_layers.html#module-qiskit_addon_mpf.backends.tenpy_layers) spiega anche in grande dettaglio. Per utilizzare questo metodo, partite dal circuito che implementa l'evoluzione temporale desiderata e create modelli che rappresentano queste operazioni dai layer del circuito corrispondente. Infine, viene creato un oggetto `Evolver` che può essere utilizzato per generare le quantità evolute nel tempo $M_{i,j}(t)$ e $L_i(t)$. Iniziamo creando l'oggetto `Evolver` corrispondente all'evoluzione temporale approssimata ([`ApproxEvolverFactory`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.dynamic.html#qiskit_addon_mpf.dynamic.ApproxEvolverFactory)) implementata dai circuiti. In particolare, prestate particolare attenzione alla variabile `order` in modo che corrispondano. Notate che nel generare i circuiti corrispondenti all'evoluzione temporale approssimata, utilizziamo valori segnaposto per il `time = 1.0` e il numero di passi di Trotter (`reps=1`). I circuiti approssimanti corretti vengono quindi prodotti dal risolutore del problema dinamico in `setup_dynamic_lse`.

In [24]:
# -------------------------Step 3-------------------------
estimator = Estimator(mode=backend)
estimator.options.default_shots = 30000

# Error suppression/mitigation
estimator.options.dynamical_decoupling.enable = True
estimator.options.twirling.enable_gates = True
estimator.options.twirling.enable_measure = True
estimator.options.twirling.num_randomizations = "auto"
estimator.options.twirling.strategy = "active-accum"
estimator.options.resilience.measure_mitigation = True
estimator.options.experimental.execution_path = "gen3-turbo"

estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 1.2, 1.4)
estimator.options.resilience.zne.extrapolator = "linear"

estimator.options.environment.job_tags = ["TUT_MPF"]

job_50 = estimator.run(
    [
        (circ, observable)
        for circ, observable in zip(transpiled_circuits, isa_observables)
    ]
)

> **Warning:** Le opzioni di `LayerwiseEvolver` che determinano i dettagli della simulazione della rete tensoriale devono essere scelte con attenzione per evitare di impostare un problema di ottimizzazione mal definito.
Quindi impostiamo l'evolver esatto (ad esempio, [`ExactEvolverFactory`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.dynamic.html#qiskit_addon_mpf.dynamic.ExactEvolverFactory)), che restituisce un oggetto [`Evolver`](https://qiskit.github.io/qiskit-addon-mpf/apidocs/qiskit_addon_mpf.backends.html#qiskit_addon_mpf.backends.Evolver) che calcola l'evoluzione temporale vera o "di riferimento". Realisticamente, approssimeremmo l'evoluzione esatta utilizzando una formula di Suzuki-Trotter di ordine superiore o un altro metodo affidabile con un piccolo passo temporale. Qui sotto, approssimiamo lo stato evoluto nel tempo esatto con una formula di Suzuki-Trotter del quarto ordine utilizzando un piccolo passo temporale `dt=0.1`, il che significa che il numero di passi di Trotter utilizzati al tempo $t$ è $k=t/dt$. Specifichiamo anche alcune opzioni di troncamento specifiche di TeNPy per limitare la dimensione massima del legame della rete tensoriale sottostante, così come i valori singolari minimi dei legami della rete tensoriale divisa. Questi parametri possono influenzare l'accuratezza del valore di aspettazione calcolato con i coefficienti MPF dinamici, quindi è importante esplorare un intervallo di valori per trovare l'equilibrio ottimale tra tempo di calcolo e accuratezza. Notate che il calcolo dei coefficienti MPF non si basa sul valore di aspettazione della PF ottenuto dall'esecuzione hardware, e pertanto può essere sintonizzato in post-elaborazione.

In [25]:
# -------------------------Step 4-------------------------
result = job_50.result()
evs = [res.data.evs for res in result]
std = [res.data.stds for res in result]

print(evs)
print(std)

[array(-0.07916195), array(-0.04479681), array(-0.2560756), array(-0.06045848)]
[array(0.04605538), array(0.10056336), array(0.14426151), array(0.04059092)]


In [26]:
exact_mpf_std = np.sqrt(
    sum([(coeff**2) * (std**2) for coeff, std in zip(mpf_coeffs, std[:3])])
)
print(
    "Exact static MPF expectation value: ",
    evs[:3] @ mpf_coeffs,
    "+-",
    exact_mpf_std,
)
approx_mpf_std = np.sqrt(
    sum(
        [
            (coeff**2) * (std**2)
            for coeff, std in zip(coeffs_approx.value, std[:3])
        ]
    )
)
print(
    "Approximate static MPF expectation value: ",
    evs[:3] @ coeffs_approx.value,
    "+-",
    approx_mpf_std,
)
dynamic_mpf_std = np.sqrt(
    sum(
        [
            (coeff**2) * (std**2)
            for coeff, std in zip(mpf_dynamic_coeffs, std[:3])
        ]
    )
)
print(
    "Dynamic MPF expectation value: ",
    evs[:3] @ mpf_dynamic_coeffs,
    "+-",
    dynamic_mpf_std,
)

Exact static MPF expectation value:  -0.5665938395816946 +- 0.3925273058119915
Approximate static MPF expectation value:  -0.25856647611537903 +- 0.164249927266166
Dynamic MPF expectation value:  -0.12667812062949296 +- 0.06059471006973169


In [27]:
sym = {3: "^", 4: "s", 6: "p"}
for k, step in enumerate(mpf_trotter_steps):
    plt.errorbar(
        k,
        evs[k],
        yerr=std[k],
        alpha=0.5,
        markersize=4,
        marker=sym[step],
        color="grey",
        label=f"{mpf_trotter_steps[k]} Trotter steps",
    )

plt.errorbar(
    3,
    evs[-1],
    yerr=std[-1],
    alpha=0.5,
    markersize=8,
    marker="x",
    color="blue",
    label="10 Trotter steps",
)

plt.errorbar(
    4,
    evs[:3] @ mpf_coeffs,
    yerr=exact_mpf_std,
    markersize=4,
    marker="o",
    color="purple",
    label="Static MPF",
)

plt.errorbar(
    5,
    evs[:3] @ coeffs_approx.value,
    yerr=approx_mpf_std,
    markersize=4,
    marker="o",
    color="orange",
    label="Approximate static MPF",
)

plt.errorbar(
    6,
    evs[:3] @ mpf_dynamic_coeffs,
    yerr=dynamic_mpf_std,
    markersize=4,
    marker="o",
    color="pink",
    label="Dynamic MPF",
)

exact_obs = -0.24384471447172074  # Calculated via Tensor Network calculation
plt.axhline(
    y=exact_obs, linestyle="--", color="red", label="Exact time-evolution"
)

plt.title(
    f"$\\langle Z_{{{L//2-1}}} Z_{{{L//2}}} \\rangle$ at time {total_time} for the different methods"
)
plt.xlabel("Method")
plt.ylabel("Expectation Value")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.grid(alpha=0.1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/multi-product-formula/extracted-outputs/64360d85-0.avif" alt="Output of the previous code cell" />

A few observations about the hardware results above:

- **Going deeper is not free on hardware.** The single-circuit baselines tell the story directly: the $k = 6$ circuit is essentially exact ($-0.256$ versus the reference $-0.244$), yet the deeper $k = 10$ baseline is *worse* ($-0.061$, off by $\sim 0.18$), not better. Once the Trotter error is already small, adding steps mostly deepens the circuit and accumulates more gate noise and decoherence. This is precisely the regime MPFs are built for: reach the accuracy of a deep circuit using only shallow constituents.

- **A small-norm MPF beats the deep single circuit.** The approximate-static MPF (capped at $\|x\|_1 \approx 2$) lands at $-0.259$, within $\sim 0.015$ of the reference and far closer than the $k = 10$ baseline. The dynamic MPF ($-0.127$) also comfortably beats that baseline. Both combine only the shallow $k_j = [3, 4, 6]$ circuits, yet recover an answer the deep single circuit could not.

- **Coefficient norm matters more than mathematical optimality.** The exact-static MPF has $\|x\|_1 = 4.66$ and is the *worst* estimator of all ($-0.567$, off by more than $0.3$): the large coefficient norm amplifies the residual gate noise, decoherence, and ZNE error on each $\langle A \rangle_{k_j}$ by roughly the same factor, overwhelming the Trotter-error cancellation it buys. Capping the norm (the approximate-static solver, $\|x\|_1 \approx 2$) removes this overwhelm and gives the best estimate &mdash; even though its coefficients no longer cancel the leading Trotter error exactly.

- **Individual shallow circuits can still be competitive.** The lone $k = 6$ constituent ($-0.256$) is itself essentially exact here &mdash; on this run it is even marginally closer than the approximate-static MPF. The catch is that you do not know in advance *which* single $k$ sits in the sweet spot of "converged but not yet noise-limited," and the safe-looking choice of simply going deeper ($k = 10$) to guarantee Trotter convergence is exactly the one that fails. The MPF gives a principled combination of shallow circuits that does not require guessing the right depth.

The practical takeaway is that on hardware, MPFs should be paired with strong error mitigation on each individual $\langle A \rangle_{k_j}$, the coefficient $L_1$-norm should be kept modest (use the approximate solver, or the dynamic MPF), and the Trotter steps $k_j$ should be chosen so that $t/k_{\min} \lesssim 1$ &mdash; here $k_{\min} = 3$ at $t = 3$ gives $t/k_{\min} = 1$, keeping the constituents inside the convergent regime where the leading-error model the static MPF relies on is valid. With those choices, the small-norm MPFs here match a converged single circuit while the naive "just go deeper" baseline does not, recovering the depth-versus-accuracy advantage shown in Ref. [\[3\]](#references). Note also that individual runs are noisy &mdash; on a different submission of the same job (or a different backend), the exact ordering can shift; the robust trends are that small-$\|x\|_1$ MPFs do well, the large-$\|x\|_1$ exact-static MPF is amplified by hardware noise, and the over-deep single circuit is noise-limited.

## Next steps
<Admonition type="tip" title="Recommendations">

If you found this work interesting, you might be interested in the following material:
- [How to choose the Trotter steps for an MPF](https://qiskit.github.io/qiskit-addon-mpf/how_tos/choose_trotter_steps.html) — practical guidance on selecting $k_j$ values to avoid instabilities
- [How to use the approximate model](https://qiskit.github.io/qiskit-addon-mpf/how_tos/using_approximate_model.html) — tuning the $L_1$-norm constraint and solver options for the approximate static MPF
- [`qiskit-addon-mpf` API reference](https://qiskit.github.io/qiskit-addon-mpf/) — full documentation for static, dynamic, and backend modules
</Admonition>

## References

\[1] Vazquez, A. C., Egger, D. J., Ochsner, D., & Woerner, S. Well-conditioned multi-product formulas for hardware-friendly Hamiltonian simulation. [Quantum, 7, 1067 (2023)](https://quantum-journal.org/papers/q-2023-07-25-1067/)

\[2] Zhuk, S., Robertson, N. F., & Bravyi, S. Trotter error bounds and dynamic multi-product formulas for Hamiltonian simulation. [Physical Review Research, 6(3), 033309 (2024)](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.6.033309)

\[3] Robertson, N. F., et al. Tensor network enhanced dynamic multiproduct formulas. [arXiv:2407.17405 (2024)](https://arxiv.org/abs/2407.17405)